Create a Spark Session

In [0]:
from pyspark.sql import SparkSession

Spark = SparkSession.builder.appName('IEBC 2022 Voter Registration statistics').getOrCreate()

Read IEBC data from the lake house

In [0]:
df_county = spark.read.csv(
    "file:/Workspace/Users/odhiambovictor86@yahoo.com/My Code/Vic_datasets/rov_per_county.csv",
    header=True,
    inferSchema=True
)

In [0]:
df_constituency = spark.read.csv("file:/Workspace/Users/odhiambovictor86@yahoo.com/My Code/Vic_datasets/rov_per_constituency.csv",
    header=True,
    inferSchema=True)

In [0]:
df_wards = spark.read.csv("file:/Workspace/Users/odhiambovictor86@yahoo.com/My Code/Vic_datasets/rov_per_caw.csv",
            header=True,
            inferSchema=True)

Verify that data has been read into memory

In [0]:
df_county.show(5)

In [0]:
df_constituency.show(5)

In [0]:
df_wards.show()

Check on schema details

In [0]:
df_county.printSchema()

In [0]:
df_constituency.printSchema() 

Transform datasets to remove the commas in the Registered Voters column

In [0]:
from pyspark.sql.functions import col, regexp_replace

df_county = df_county.withColumn(
    "REGISTERED_VOTERS",
    regexp_replace(col("REGISTERED_VOTERS"), ",", "").cast("long")
)

In [0]:
from pyspark.sql.functions import col, regexp_replace

df_constituency = df_constituency.withColumn(
    "REGISTERED_VOTERS",
    regexp_replace(col("REGISTERED_VOTERS"), ",", "").cast("long")
)

In [0]:
from pyspark.sql.functions import col, regexp_replace

df_wards = df_wards.withColumn(
    "REGISTERED_VOTERS",
    regexp_replace(col("REGISTERED_VOTERS"), ",", "").cast("long")
)

Write the data into a table so that it can be queried using Spark SQL

In [0]:
df_constituency.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("constituencies")

In [0]:
df_county.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("county")

In [0]:
df_wards.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("wards")

In [0]:
df_wards = df_wards.withColumnRenamed("County Code", "COUNTY_CODE") \
                   .withColumnRenamed("County Name", "COUNTY_NAME") \
                   .withColumnRenamed("Const.  Name", "CONST_NAME")

Running SQL queries over the created tables

In [0]:
spark.sql("SELECT * FROM county order by REGISTERED_VOTERS DESC").show()

In [0]:
spark.sql("SELECT * FROM constituencies order by REGISTERED_VOTERS DESC").show()


In [0]:
spark.sql("SELECT * FROM wards order by REGISTERED_VOTERS DESC").show()